# AI-Powered Engineering Study Assistant (Capstone)

Full RAG + Agent + Memory + Tool

## Capstone Plan

Domain: Engineering Study Assistant

User: Engineering students

Success: Accurate RAG answers, memory support, tool usage

In [ ]:
!pip install langgraph langchain-groq chromadb sentence-transformers python-dotenv -q

In [ ]:
import os
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq


In [ ]:
DOCUMENTS = [
{'id':'1','text':'Ohm law V=IR'},
{'id':'2','text':'Kirchhoff laws circuits'},
{'id':'3','text':'SHM periodic motion'},
{'id':'4','text':'Thermodynamics heat'},
{'id':'5','text':'Wave motion energy'},
{'id':'6','text':'Capacitance charge'},
{'id':'7','text':'Inductance magnetic'},
{'id':'8','text':'Semiconductors electronics'},
{'id':'9','text':'Logic gates'},
{'id':'10','text':'Control systems'}
]

In [ ]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
client = chromadb.Client()
collection = client.create_collection('kb')
texts=[d['text'] for d in DOCUMENTS]
ids=[d['id'] for d in DOCUMENTS]
emb=embedder.encode(texts).tolist()
collection.add(documents=texts, embeddings=emb, ids=ids)

In [ ]:
groq_key = os.getenv('GROQ_API_KEY','')
llm = ChatGroq(model='llama-3.1-8b-instant', groq_api_key=groq_key)


In [ ]:
def retrieve(query):
    q_emb = embedder.encode([query]).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=2)
    return res['documents'][0]


In [ ]:
def answer(query):
    docs = retrieve(query)
    context = '\n'.join(docs)
    prompt = f'Answer using context:\n{context}\nQ:{query}'
    return llm.invoke(prompt).content


In [ ]:
def tool_time():
    import datetime
    return str(datetime.datetime.now())


In [ ]:
memory = {}
def chat(user, query):
    if 'time' in query.lower():
        return tool_time()
    memory[user] = query
    return answer(query)


In [ ]:
print(chat('nikhil','What is Ohm law?'))
print(chat('nikhil','what is time now?'))


## Reflection

This system integrates Retrieval-Augmented Generation (RAG) with a Large Language Model (LLM) to provide accurate and context-based answers. Instead of relying on general knowledge, the model retrieves relevant documents from a structured knowledge base using embeddings stored in ChromaDB.

The use of memory allows the system to retain user interactions, enabling more personalized and context-aware conversations. Additionally, tool integration, such as the date and time function, enhances the system’s capability to handle real-time queries beyond static knowledge retrieval.

Overall, the combination of RAG, memory, and tool usage makes the system more reliable, reduces hallucination, and improves the quality of responses. This architecture demonstrates how modern AI systems can be designed to solve domain-specific problems effectively.